# AI-Powered Talent Ranking and Interactive Reranking

This notebook is a mentor-friendly version of the talent-ranking pipeline.

It is organized as:

1. setup and configuration
2. data loading and preprocessing
3. reusable ranking functions
4. baseline semantic + lexical + location ranking
5. relevance gate
6. feedback-based reranking
7. evaluation
8. export

## 1. Setup

If the environment does not already contain the required packages, uncomment and run the installation line below once.

In [25]:
# %pip install pandas numpy scikit-learn sentence-transformers geopy nltk

In [26]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import CrossEncoder
from geopy.distance import geodesic

nltk.download("stopwords", quiet=True)
STOP_WORDS = set(stopwords.words("english"))

pd.set_option("display.max_colwidth", 100)

## 2. Configuration

Change only this cell for a new experiment.

- `SEARCH_QUERIES` defines the original recruiter search.
- `POSITION_COORDS` can be set to `(latitude, longitude)` if location should affect ranking.
- `GOOD_IDS` and `BAD_IDS` provide explicit recruiter feedback.

Leaving `POSITION_COORDS = None` assigns every candidate a neutral location score so the notebook can still be run end-to-end.

In [27]:
SEARCH_QUERIES = [
    "Aspiring human resources",
    "seeking human resources",
]

# Example format: (43.6532, -79.3832)
POSITION_COORDS = None

# Edit these and rerun the feedback section to test human-in-the-loop reranking.
GOOD_IDS = []
BAD_IDS = []

TOP_K = 10

## 3. Load and preprocess the candidate data

This notebook requires the original csv, which is not in the repo for data security reasons.

The preprocessing below then reproduces the core job-title cleaning used in `eda.py`.

In [28]:
DATA_PATH = Path("potential-talents - Aspiring human resources - seeking human resources.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "potential-talents - Aspiring human resources - seeking human resources.csv was not found. Run the notebook from the repository root."
    )

data = pd.read_csv(DATA_PATH)
data = data.drop(columns=["Unnamed: 0"], errors="ignore")

# Remove exact duplicate candidate records using the original project rule.
data = data.drop_duplicates(subset=["job_title", "location", "connection"]).copy()

# Clean titles: letters/spaces only, lowercase, then remove English stopwords.
data["job_title"] = (
    data["job_title"]
    .fillna("")
    .astype(str)
    .str.replace(r"[^a-zA-Z ]", "", regex=True)
    .str.lower()
)

data["job_title"] = data["job_title"].apply(
    lambda text: " ".join(
        word for word in text.split()
        if word.casefold() not in STOP_WORDS
    )
)

# Convert LinkedIn-style connection counts such as '500+' into numeric values.
data["connection"] = (
    data["connection"]
    .astype(str)
    .str.strip()
    .str.replace("+", "", regex=False)
)

data["connection"] = pd.to_numeric(data["connection"], errors="coerce").fillna(0)
data["id"] = data["id"].astype(str)

print(f"Working candidate rows: {len(data)}")
data.head()

Working candidate rows: 53


,id,job_title,location,connection,fit
0,1,ct bauer college business graduate magna cum laude aspiring human resources professional,"Houston, Texas",85,NaN
1,2,native english teacher epik english program korea,Kanada,500,NaN
2,3,aspiring human resources professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,people development coordinator ryan,"Denton, Texas",500,NaN
4,5,advisory board member celal bayar university,"İzmir, Türkiye",500,NaN


## 4. Load the semantic model and initialize evaluation state

`cross-encoder/stsb-roberta-base` is used as the semantic sentence-pair model.

In [29]:
ce_model = CrossEncoder("cross-encoder/stsb-roberta-base")

feedback_metric_columns = [
    "Change Number",
    "Good Feedback Count",
    "Bad Feedback Count",
    "Mean Good Model Rank",
    "Mean Bad Model Rank",
    f"Good Recall@{TOP_K}",
    "Good-Bad Pairwise Accuracy",
    "Good-Bad Score Margin",
]

feedback_metrics_history = pd.DataFrame(columns=feedback_metric_columns)

## 5. Reusable ranking functions

In [30]:
def normalize_for_column(text):
    return (
        str(text)
        .strip()
        .lower()
        .replace(" ", "_")[:30]
    )


def make_final_scores(db, desc):
    score_columns = [
        column for column in db.columns
        if column.startswith(f"score_{desc}_")
    ]

    if not score_columns:
        return db

    db[f"final_{desc}_score_mean"] = db[score_columns].mean(axis=1)
    db[f"final_{desc}_score_max"] = db[score_columns].max(axis=1)
    return db


def fit_tfidf_to_query(query_input, title_col, db, desc):
    queries = [
        str(query).strip()
        for query in query_input
        if str(query).strip()
    ]

    if not queries:
        return db

    candidate_titles = (
        db[title_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .replace("", "unknown")
    )

    combined_corpus = candidate_titles.tolist() + queries

    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        sublinear_tf=True,
        norm="l2",
    )

    tfidf_matrix = vectorizer.fit_transform(combined_corpus)
    candidate_matrix = tfidf_matrix[:len(db)]
    query_matrix = tfidf_matrix[len(db):]

    similarities = cosine_similarity(candidate_matrix, query_matrix)

    for query_index, query in enumerate(queries):
        column_name = f"score_{desc}_{normalize_for_column(query)}"
        db[column_name] = similarities[:, query_index]

    return make_final_scores(db, desc)


def fit_model_to_query(query_input, title_col, model, db, desc):
    for query in query_input:
        pairs = [[query, text] for text in db[title_col]]

        scores = model.predict(
            pairs,
            batch_size=32,
            show_progress_bar=False,
        )

        similarity_scores = np.clip(
            np.asarray(scores, dtype=float),
            0.0,
            1.0,
        )

        column_name = f"score_{desc}_{normalize_for_column(query)}"
        db[column_name] = similarity_scores

    if len(query_input) > 0:
        db = make_final_scores(db, desc)

    return db


def add_distance_from_preferred_loc(db, loc_cols, loc_coords):
    lat_col, lon_col = loc_cols

    # Notebook-friendly behavior: use a neutral location score when no job
    # coordinates are provided.
    if loc_coords is None:
        db["distance_from_loc"] = np.nan
        db["location_score"] = 0.5
        return db

    db["distance_from_loc"] = db.apply(
        lambda row: geodesic(
            loc_coords,
            (row[lat_col], row[lon_col])
        ).km,
        axis=1,
    )

    minimum = db["distance_from_loc"].min()
    maximum = db["distance_from_loc"].max()

    if maximum == minimum:
        db["location_score"] = 1.0
    else:
        db["location_score"] = 1 - (
            (db["distance_from_loc"] - minimum)
            / (maximum - minimum)
        )

    return db


def combine_similarity_scores(
    db,
    ce_desc,
    tfidf_desc,
    output_desc,
    dist_col="location_score",
    ce_weight=0.65,
    tfidf_weight=0.30,
    loc_weight=0.05,
):
    total_weight = ce_weight + tfidf_weight + loc_weight

    if total_weight <= 0:
        raise ValueError("Similarity weights must sum to a positive value.")

    for statistic in ["mean", "max"]:
        ce_column = f"final_{ce_desc}_score_{statistic}"
        tfidf_column = f"final_{tfidf_desc}_score_{statistic}"
        combined_column = f"final_{output_desc}_score_{statistic}"

        db[combined_column] = (
            loc_weight * db[dist_col]
            + ce_weight * db[ce_column]
            + tfidf_weight * db[tfidf_column]
        ) / total_weight

    return db


def add_initial_relevance_gate(
    db,
    ce_column="final_ce_base_score_mean",
    tfidf_column="final_tfidf_base_score_max",
    semantic_floor_quantile=0.30,
):
    semantic_floor = db[ce_column].quantile(semantic_floor_quantile)

    db["passes_initial_gate"] = (
        (db[ce_column] >= semantic_floor)
        | (db[tfidf_column] > 0)
    )

    return db, semantic_floor

In [31]:
def update_feedback_metrics(
    metrics_table,
    db,
    id_col,
    score_col,
    good_ids,
    bad_ids,
    change_number,
    k=10,
):
    ranked = (
        db.sort_values(score_col, ascending=False)
        .reset_index(drop=True)
        .copy()
    )

    ranked["evaluation_rank"] = np.arange(len(ranked)) + 1
    indexed = ranked.set_index(id_col)

    available_good = [
        candidate_id for candidate_id in good_ids
        if candidate_id in indexed.index
    ]

    available_bad = [
        candidate_id for candidate_id in bad_ids
        if candidate_id in indexed.index
    ]

    good_scores = indexed.loc[available_good, score_col].to_numpy(dtype=float)
    bad_scores = indexed.loc[available_bad, score_col].to_numpy(dtype=float)
    good_ranks = indexed.loc[available_good, "evaluation_rank"].to_numpy(dtype=float)
    bad_ranks = indexed.loc[available_bad, "evaluation_rank"].to_numpy(dtype=float)

    top_k_ids = set(ranked.head(k)[id_col])

    if available_good:
        good_recall_at_k = (
            len(top_k_ids.intersection(available_good))
            / len(available_good)
        )
        mean_good_rank = good_ranks.mean()
    else:
        good_recall_at_k = np.nan
        mean_good_rank = np.nan

    if available_bad:
        mean_bad_rank = bad_ranks.mean()
    else:
        mean_bad_rank = np.nan

    if available_good and available_bad:
        greater_than = good_scores[:, None] > bad_scores[None, :]
        equal_to = good_scores[:, None] == bad_scores[None, :]
        pairwise_accuracy = np.mean(greater_than + 0.5 * equal_to)
        score_margin = good_scores.mean() - bad_scores.mean()
    else:
        pairwise_accuracy = np.nan
        score_margin = np.nan

    new_row = pd.DataFrame([{
        "Change Number": change_number,
        "Good Feedback Count": len(available_good),
        "Bad Feedback Count": len(available_bad),
        "Mean Good Model Rank": mean_good_rank,
        "Mean Bad Model Rank": mean_bad_rank,
        f"Good Recall@{k}": good_recall_at_k,
        "Good-Bad Pairwise Accuracy": pairwise_accuracy,
        "Good-Bad Score Margin": score_margin,
    }])

    return pd.concat([metrics_table, new_row], ignore_index=True)


def rerank_from_feedback(
    db,
    good_ids,
    bad_ids,
    ce_model,
    id_col="id",
    title_col="job_title",
    base_col="final_base_score_mean",
):
    db = db.copy()

    # Make IDs consistent with the notebook's string ID convention.
    good_ids = list(dict.fromkeys(str(x) for x in good_ids))
    bad_ids = list(dict.fromkeys(str(x) for x in bad_ids))

    valid_ids = set(db[id_col])
    good_ids = [x for x in good_ids if x in valid_ids]
    bad_ids = [x for x in bad_ids if x in valid_ids]

    # If an ID appears in both lists, the most conservative choice here is to
    # remove it from both until the recruiter resolves the conflict.
    overlap = set(good_ids).intersection(bad_ids)
    good_ids = [x for x in good_ids if x not in overlap]
    bad_ids = [x for x in bad_ids if x not in overlap]

    good_titles = (
        db.loc[db[id_col].isin(good_ids), title_col]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    bad_titles = (
        db.loc[db[id_col].isin(bad_ids), title_col]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    if good_titles:
        db = fit_model_to_query(
            good_titles, title_col, ce_model, db, "ce_good"
        )
        db = fit_tfidf_to_query(
            good_titles, title_col, db, "tfidf_good"
        )
        db = combine_similarity_scores(
            db,
            ce_desc="ce_good",
            tfidf_desc="tfidf_good",
            output_desc="good",
            ce_weight=0.70,
            tfidf_weight=0.30,
        )

    if bad_titles:
        db = fit_model_to_query(
            bad_titles, title_col, ce_model, db, "ce_bad"
        )
        db = fit_tfidf_to_query(
            bad_titles, title_col, db, "tfidf_bad"
        )
        db = combine_similarity_scores(
            db,
            ce_desc="ce_bad",
            tfidf_desc="tfidf_bad",
            output_desc="bad",
            ce_weight=0.70,
            tfidf_weight=0.30,
        )

    weighted_score = 0.45 * db[base_col]
    active_weight = 0.45

    # Corrected from the script's `connections` typo: the data column is singular.
    db["norm_connections"] = (db["connection"] / 500).clip(lower=0, upper=1)
    weighted_score += 0.05 * db["norm_connections"]
    active_weight += 0.05

    if "final_good_score_max" in db.columns:
        weighted_score += 0.35 * db["final_good_score_max"]
        active_weight += 0.35

    if "final_bad_score_max" in db.columns:
        bad_avoidance = 1 - db["final_bad_score_max"]
        weighted_score += 0.15 * bad_avoidance
        active_weight += 0.15

    db["model_fit_score"] = weighted_score / active_weight
    db["model_rank"] = db["model_fit_score"].rank(
        ascending=False,
        method="min",
    )

    return db, good_ids, bad_ids

## 6. Baseline ranking

The baseline combines:

- CrossEncoder semantic similarity
- TF-IDF lexical similarity
- optional geographic proximity

In [32]:
baseline = data.copy()

baseline = add_distance_from_preferred_loc(
    baseline,
    ["latitude", "longitude"],
    POSITION_COORDS,
)

baseline = fit_model_to_query(
    SEARCH_QUERIES,
    "job_title",
    ce_model,
    baseline,
    "ce_base",
)

baseline = fit_tfidf_to_query(
    SEARCH_QUERIES,
    "job_title",
    baseline,
    "tfidf_base",
)

baseline = combine_similarity_scores(
    baseline,
    ce_desc="ce_base",
    tfidf_desc="tfidf_base",
    output_desc="base",
    dist_col="location_score",
    ce_weight=0.70,
    tfidf_weight=0.30,
    loc_weight=0.05,
)

baseline["rank"] = baseline["final_base_score_mean"].rank(
    ascending=False,
    method="min",
)

baseline = baseline.sort_values("rank").reset_index(drop=True)

baseline[
    [
        "id",
        "job_title",
        "location",
        "connection",
        "final_ce_base_score_mean",
        "final_tfidf_base_score_mean",
        "location_score",
        "final_base_score_mean",
        "rank",
    ]
].head(15)

,id,job_title,location,connection,final_ce_base_score_mean,final_tfidf_base_score_mean,location_score,final_base_score_mean,rank
0,99,seeking human resources position,"Las Vegas, Nevada Area",48,0.829774,0.443396,0.5,0.703677,1.0
1,28,seeking human resources opportunities,"Chicago, Illinois",390,0.828719,0.443396,0.5,0.702973,2.0
2,3,aspiring human resources professional,"Raleigh-Durham, North Carolina Area",44,0.742720,0.486305,0.5,0.657901,3.0
3,97,aspiring human resources professional,"Kokomo, Indiana Area",71,0.742720,0.486305,0.5,0.657901,3.0
4,6,aspiring human resources specialist,Greater New York City Area,1,0.761726,0.417632,0.5,0.650951,5.0
5,73,aspiring human resources manager seeking internship human resources,"Houston, Texas Area",7,0.708151,0.405623,0.5,0.611802,6.0
6,10,seeking human resources hris generalist positions,Greater Philadelphia Area,500,0.661453,0.281802,0.5,0.545293,7.0
7,27,aspiring human resources management student seeking internship,"Houston, Texas Area",500,0.641932,0.305260,0.5,0.538982,8.0
8,82,aspiring human resources professional energetic teamfocused leader,"Austin, Texas Area",174,0.667052,0.233692,0.5,0.535280,9.0
9,94,seeking human resources opportunities open travel relocation,Amerika Birleşik Devletleri,415,0.644561,0.247980,0.5,0.524368,10.0


## 7. Initial relevance gate

Candidates below the 30th-percentile CrossEncoder semantic floor are flagged unless they have positive TF-IDF overlap with the query.

In [33]:
baseline, semantic_floor = add_initial_relevance_gate(baseline)

print(f"Semantic floor: {semantic_floor:.4f}")
print(baseline["passes_initial_gate"].value_counts())

initially_excluded = baseline.loc[
    ~baseline["passes_initial_gate"],
    [
        "id",
        "job_title",
        "final_ce_base_score_mean",
        "final_tfidf_base_score_max",
        "final_base_score_mean",
    ],
]

initially_excluded.head(20)

Semantic floor: 0.1203
passes_initial_gate
True     37
False    16
Name: count, dtype: int64


,id,job_title,final_ce_base_score_mean,final_tfidf_base_score_max,final_base_score_mean
37,80,junior mes engineer information systems,0.119448,0.0,0.103442
38,93,admissions representative community medical center long beach,0.101341,0.0,0.091370
39,98,student,0.091157,0.0,0.084581
40,5,advisory board member celal bayar university,0.073275,0.0,0.072660
41,12,svp chro marketing communications csr officer engie houston woodlands energy gphr sphr,0.069535,0.0,0.070166
42,87,bachelor science biology victoria university wellington,0.064436,0.0,0.066767
43,96,student indiana university kokomo business management retail manager delphi hardware paint,0.055999,0.0,0.061142
44,91,lead official western illinois university,0.051807,0.0,0.058348
45,95,student westfield state university,0.026935,0.0,0.041766
46,2,native english teacher epik english program korea,0.025463,0.0,0.040785


## 8. Feedback-based reranking

Edit `GOOD_IDS` and `BAD_IDS` in the configuration cell, then rerun this section.

The reranking model uses:

- 45% baseline relevance
- 5% normalized connection count
- 35% similarity to good examples (when present)
- 15% avoidance of bad examples (when present)

The active weights are renormalized when good or bad examples are missing.

In [34]:
reranked, good_ids_used, bad_ids_used = rerank_from_feedback(
    baseline,
    GOOD_IDS,
    BAD_IDS,
    ce_model,
)

# Evaluate BEFORE forcing recruiter-labelled candidates to 1 or 0.
feedback_metrics_history = update_feedback_metrics(
    metrics_table=feedback_metrics_history,
    db=reranked,
    id_col="id",
    score_col="model_fit_score",
    good_ids=good_ids_used,
    bad_ids=bad_ids_used,
    change_number=len(feedback_metrics_history) + 1,
    k=TOP_K,
)

# Preserve direct recruiter decisions in the final displayed ranking.
reranked["final_fit_with_weights_mean"] = reranked["model_fit_score"].copy()

reranked.loc[
    reranked["id"].isin(good_ids_used),
    "final_fit_with_weights_mean"
] = 1.0

reranked.loc[
    reranked["id"].isin(bad_ids_used),
    "final_fit_with_weights_mean"
] = 0.0

reranked["rank"] = reranked["final_fit_with_weights_mean"].rank(
    ascending=False,
    method="min",
)

reranked = reranked.sort_values("rank").reset_index(drop=True)

reranked[
    [
        "id",
        "job_title",
        "connection",
        "final_base_score_mean",
        "model_fit_score",
        "final_fit_with_weights_mean",
        "rank",
    ]
].head(15)

,id,job_title,connection,final_base_score_mean,model_fit_score,final_fit_with_weights_mean,rank
0,28,seeking human resources opportunities,390,0.702973,0.710676,0.710676,1.0
1,99,seeking human resources position,48,0.703677,0.642909,0.642909,2.0
2,97,aspiring human resources professional,71,0.657901,0.606310,0.606310,3.0
3,3,aspiring human resources professional,44,0.657901,0.600910,0.600910,4.0
4,10,seeking human resources hris generalist positions,500,0.545293,0.590764,0.590764,5.0
5,6,aspiring human resources specialist,1,0.650951,0.586056,0.586056,6.0
6,27,aspiring human resources management student seeking internship,500,0.538982,0.585083,0.585083,7.0
7,94,seeking human resources opportunities open travel relocation,415,0.524368,0.554931,0.554931,8.0
8,73,aspiring human resources manager seeking internship human resources,7,0.611802,0.552022,0.552022,9.0
9,75,nortia staffing seeking human resources payroll administrative professionals,500,0.470769,0.523692,0.523692,10.0


## 9. Evaluation

If no good/bad IDs were entered, the feedback metrics that require labels will be `NaN`. Add IDs in the configuration cell and rerun the feedback section to evaluate a feedback round.

In [35]:
feedback_metrics_history

,Change Number,Good Feedback Count,Bad Feedback Count,Mean Good Model Rank,Mean Bad Model Rank,Good Recall@10,Good-Bad Pairwise Accuracy,Good-Bad Score Margin
0,1,0,0,NaN,NaN,NaN,NaN,NaN


### Reference development result

The saved development run supplied with the project used:

- 2 good feedback examples
- 1 bad feedback example

and produced:

- Mean Good Model Rank: **4.5**
- Mean Bad Model Rank: **34.0**
- Good Recall@10: **1.0**
- Good-Bad Pairwise Accuracy: **1.0**
- Good-Bad Score Margin: **0.3255**

Those saved results predate the newest location/connection changes, so they should be treated as a reference rather than as the final evaluation of the updated pipeline.

## 10. Export results

In [36]:
reranked.to_csv("trial_ranking.csv", index=False)

feedback_metrics_history.to_csv(
    "feedback_metrics_history.csv",
    index=False,
)

only_value_and_rank = reranked.loc[:, ["id", "job_title", "rank"]]
only_value_and_rank.to_csv("only_value_and_rank.csv", index=False)

print("Saved trial_ranking.csv")
print("Saved feedback_metrics_history.csv")
print("Saved only_value_and_rank.csv")

Saved trial_ranking.csv
Saved feedback_metrics_history.csv
Saved only_value_and_rank.csv


## Suggested next experiments

1. Add recruiter-labelled examples across several feedback rounds.
2. Rerun the saved evaluation metrics after enabling a real job location.
3. Test alternative component weights rather than relying only on hand-selected weights.
4. Evaluate whether connection count improves ranking quality and whether it introduces unwanted bias.
5. Add a larger labelled benchmark so ranking quality can be evaluated beyond the handful of manually selected candidates.